In [1]:
#Notebook description

# this notebook is intended to track and price commodities based on the demand and supply data

In [2]:
#DOUBLE CHECK HERE THAT YOUR ARE ACCOUNTING FOR ALL INDUSTRIES

In [3]:
#Load libraries 
import logging
logger = logging.getLogger('yfinance')
logger.disabled = True
logger.propagate = False
import sys
import sys
from dotenv import load_dotenv
import os
from quickfs import QuickFS

cwd = os.getcwd()           
drive = os.path.splitdrive(cwd)[0]
project_path = drive + r"\Coding Projects\Investment Analysis"
sys.path.append(r"d:\Coding Projects\Investment Analysis")
load_dotenv(drive + r"\Coding Projects\Investment Analysis\.env")
quickfs_api_key = os.getenv("API_QUICKFS")
client = QuickFS(quickfs_api_key)

# Load libraries
from Quantapp.visualization import Plotter
from Quantapp.analytics import TimeSeriesAnalytics as Rolling
from Quantapp.data import MacroDataClient
from Quantapp.data import MarketDataClient
from Quantapp.data import GICSDataClient
from Quantapp.data import CompanyDataClient
from Quantapp.visualization import PieChartPlotter
from Quantapp.visualization import BarChartPlotter
from Quantapp.visualization import LineChartPlotter

import numpy as np
import json
import os
import pandas as pd
import yfinance as yf
from statsmodels.tsa.stattools import coint
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from plotly.subplots import make_subplots
from datetime import datetime
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.graph_objects as go
import plotly.graph_objects as go
import pandas as pd
from plotly.subplots import make_subplots

#shut down warnings
import warnings
warnings.filterwarnings("ignore")


qc = Rolling()
qp = Plotter()
qe = MacroDataClient()
market_data = MarketDataClient()

gics_data = GICSDataClient(client=client, save_path=project_path)

pieChartPlotter = PieChartPlotter()
barChartPlotter = BarChartPlotter()
LineChartPlotter = LineChartPlotter()

In [4]:
#parameters
import yfinance as yf
import matplotlib.pyplot as plt

'''
#valid parameters for 'sector'
"Communication Services"
"Consumer Discretionary"
"Consumer Staples"
"Energy"
"Financials"
"Health Care"
"Industrials"
"Information Technology"
"Materials"
"Real Estate"
"Utilities"
'''
sector = "Communication Services"
period = "20y"
interval = "1d"
sector_gics_code       = gics_data.name_to_gics(sector)
capitalization = "Large Cap"
children = gics_data.retrieve_children(sector_gics_code)
industry_group_codes = [child['code'] for child in children]


In [5]:
#SIZE & IMPACT: DOWNLOAD
#plot historical market cap weighing for each industry group in the sector

#retrieve market cap weights for industry groups in the sector
industry_group_market_cap_weights = gics_data.retrieve_market_cap_children(sector_gics_code, capitalization=capitalization)
industry_group_names = [child['name'] for child in children]
industry_group_code_name_dict = {children[i]['code']: industry_group_names[i] for i in range(len(children))}
industry_group_market_cap_weights.rename(columns=industry_group_code_name_dict, inplace=True)

#retrieve enterprise value for each industry group in the sector
enterprise_value = gics_data.retrieve_fundamental_data_children(parent_gics_code=sector_gics_code, 
                                                                capitalization='Large Cap', 
                                                                data_type='quarterly',
                                                                as_weights=False,
                                                                statement_type='computed',
                                                                metric='enterprise_value',
                                                                aggregation_method='sum',
                                                                filtering_method=None,
                                                                truncate_below_zero=False,
                                                                should_update=False)
#rename columns to industry group names
enterprise_value.rename(columns=industry_group_code_name_dict, inplace=True)
display(enterprise_value.head())



revenue = gics_data.retrieve_fundamental_data_children(parent_gics_code=sector_gics_code, 
                                                       capitalization='Large Cap',
                                                       data_type='quarterly',
                                                       as_weights=False,
                                                       statement_type='income_statement',
                                                       metric='revenue',
                                                       aggregation_method='sum',
                                                       filtering_method=None,
                                                       truncate_below_zero=False,
                                                       should_update=False)

#rename columns to industry group names
revenue.rename(columns=industry_group_code_name_dict, inplace=True)



ConnectTimeout: HTTPSConnectionPool(host='public-api.quickfs.net', port=443): Max retries exceeded with url: /v1/data/all-data/T (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000016275FE2EA0>, 'Connection to public-api.quickfs.net timed out. (connect timeout=300)'))

In [ ]:
#SIZE & IMPACT: PLOT
#stacked bar chart of revenue for all industry groups in the sector
fig = px.bar(revenue, x=revenue.index, y=revenue.columns, title=f'Revenue for Industry Groups in Sector: {sector}')
fig.update_layout(xaxis_title='Period End Date', yaxis_title='Revenue (in millions USD)')
fig.show()

#stacked bar chart of enterprise value for all industry groups in the sector
fig = px.bar(enterprise_value, x=enterprise_value.index, y=enterprise_value.columns, title=f'Enterprise Value for Industry Groups in Sector: {sector}')
fig.update_layout(xaxis_title='Period End Date', yaxis_title='Enterprise Value (in millions USD)')
fig.show()

barChartPlotter.plot_market_cap_weights(industry_group_market_cap_weights, f"Market Cap Weights in Sector: {sector} ({sector_gics_code})")
    

In [ ]:
#Market Cap Pie Chart for Communications Sector
pieChartPlotter.plot_sector_market_cap(sector=sector)

In [ ]:
#Market Cap Bar Chart for Communications Sector
barChartPlotter.plot_sector_market_cap(sector=sector).show()

In [ ]:
#compute sector, industry group market cap weighted indices
# hierarchy dictionaries of GICS codes and their direct children
industry_group_codes
industry_codes = {}
sub_industry_codes = {}

#save industry codes for each industry group
for industry_group_code in industry_group_codes:
    industry_group = gics_data.retrieve_children(industry_group_code)
    #save a dictionary of industry codes for each industry group
    industry_codes[industry_group_code] = [industry['code'] for industry in industry_group]

#save sub-industry codes for each industry
for industry_group_code, industries in industry_codes.items():
    for industry_code in industries:
        sub_industries = gics_data.retrieve_children(industry_code)
        sub_industry_codes[industry_code] = [sub_industry['code'] for sub_industry in sub_industries]


sector_index = gics_data.calculate_weighted_index(sector_gics_code, capitalization='Large Cap', period=period, interval=interval)
sector_index.name = sector + " Sector index"

#calculate industry group indices and aggregate them into a dataframe
industry_group_indices = {}
for industry_group_code in industry_group_codes:
    industry_group_index = gics_data.calculate_weighted_index(industry_group_code, capitalization='Large Cap', period=period, interval=interval)
    name = gics_data.gics_to_name(industry_group_code)['Industry Group'] + " Industry Group index"
    industry_group_indices[name] = industry_group_index
industry_group_indices_df = pd.DataFrame(industry_group_indices)


In [ ]:
#combine sector index and industry group indices into a single dataframe
combined_indices_df = pd.concat([sector_index, industry_group_indices_df], axis=1)
#display(combined_indices_df)

sector_index = combined_indices_df['Communication Services Sector index']
industry_group_index = combined_indices_df['Telecommunication Services Industry Group index']

#PLOT 200 DAY ROLLING SHARPE RATIO OF COMBINED INDICES BY HAND AND PLOT ON ONE CHART

sharpe_ratios = combined_indices_df.pct_change().rolling(window=200).mean() / combined_indices_df.pct_change().rolling(window=200).std() * np.sqrt(252)
fig = px.line(sharpe_ratios, x=sharpe_ratios.index,
                y=sharpe_ratios.columns,
                title=f"200 Day Rolling Sharpe Ratios - {sector} Sector and Industry Groups",
                labels={"value": "Sharpe Ratio", "index": "Date"})
fig.show()

#plot 200 day rolling sharpe diff between sector and industry group indices
sharpe_diff = sharpe_ratios.subtract(sharpe_ratios['Communication Services Sector index'], axis=0)
sharpe_diff = sharpe_diff.drop(columns=['Communication Services Sector index']) 
fig = px.line(sharpe_diff, x=sharpe_diff.index,
                y=sharpe_diff.columns,
                title=f"Sector minus Industry Group 200 Day Rolling Sharpe Ratio Difference - {sector} Sector",
                labels={"value": "Sharpe Ratio Difference", "index": "Date"})
#add horizontal line at y=0
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.show()